In [7]:
import pandas as pd
from datetime import datetime
import geopandas as gpd
from css_geodata_service.extract.system_tools import extract_buildings_from_pbf_to_osmxml
from css_geodata_service.utils import get_osm_data_weser_bergland_path, get_polygon_weser_bergland
import shapely
from pathlib import Path
import os
from css_geodata_service.extract.python_tools import filter_gdf_from_from_gdf_filter_own
import logging
import fiona

logging.basicConfig(level=logging.INFO)

In [ ]:
"""
 TODO: do
 - load population data csv file
 - load buildings
 - load polygon shape to get affected area
 - filter affected buildings and households / population
 - save / visualize
 :return:
"""

In [9]:
# bounds#
polygon = get_polygon_weser_bergland()

# osm data path
osm_data_path = get_osm_data_weser_bergland_path()

In [56]:
# load population data csv file
start_reading_csv_pandas = datetime.now()
population_pandas = pd.read_csv("~/GeoData/Population/population_weser_bergland.csv")
passed_reading_csv_pandas = datetime.now() - start_reading_csv_pandas
passed_reading_csv_pandas

datetime.timedelta(seconds=10, microseconds=951137)

In [57]:
population_pandas.head()

,person_id,cell_id,household_id,household_size,household_type,age,sex,geometry
0,0,100mN33211E42447,0,3,Paare mit Kind(ern),50 - 64,Weiblich,POINT (8.864977632019484 52.99372822383581)
1,1,100mN33211E42447,0,3,Paare mit Kind(ern),18 - 29,Männlich,POINT (8.864977632019484 52.99372822383581)
2,2,100mN33211E42447,0,3,Paare mit Kind(ern),Unter 18,Weiblich,POINT (8.864977632019484 52.99372822383581)
3,3,100mN33200E42427,1,3,Paare mit Kind(ern),18 - 29,Weiblich,POINT (8.834359450000001 52.9834367)
4,4,100mN33200E42427,1,3,Paare mit Kind(ern),30 - 49,Männlich,POINT (8.834359450000001 52.9834367)


In [ ]:
# load flooded area as dataframe
affected_area = gpd.read_file(
    "~/GeoData/Flooding/HazardAreas/nz_hazardArea_fluival_H-DE_cropped_weser_bergland.geojson"
)

In [64]:
population_geopandas = gpd.GeoDataFrame(
    population_pandas, geometry=population_pandas["geometry"].apply(shapely.wkt.loads)
)

In [65]:
population_geopandas.head()

,person_id,cell_id,household_id,household_size,household_type,age,sex,geometry
0,0,100mN33211E42447,0,3,Paare mit Kind(ern),50 - 64,Weiblich,POINT (8.86498 52.99373)
1,1,100mN33211E42447,0,3,Paare mit Kind(ern),18 - 29,Männlich,POINT (8.86498 52.99373)
2,2,100mN33211E42447,0,3,Paare mit Kind(ern),Unter 18,Weiblich,POINT (8.86498 52.99373)
3,3,100mN33200E42427,1,3,Paare mit Kind(ern),18 - 29,Weiblich,POINT (8.83436 52.98344)
4,4,100mN33200E42427,1,3,Paare mit Kind(ern),30 - 49,Männlich,POINT (8.83436 52.98344)


In [8]:
households = population_pandas.groupby("household_id")

In [7]:
households.head()

,person_id,cell_id,household_id,household_size,household_type,age,sex,location
0,0,100mN33211E42447,0,3,Paare mit Kind(ern),50 - 64,Weiblich,POINT (8.864977632019484 52.99372822383581)
1,1,100mN33211E42447,0,3,Paare mit Kind(ern),18 - 29,Männlich,POINT (8.864977632019484 52.99372822383581)
2,2,100mN33211E42447,0,3,Paare mit Kind(ern),Unter 18,Weiblich,POINT (8.864977632019484 52.99372822383581)
3,3,100mN33200E42427,1,3,Paare mit Kind(ern),18 - 29,Weiblich,POINT (8.834359450000001 52.9834367)
4,4,100mN33200E42427,1,3,Paare mit Kind(ern),30 - 49,Männlich,POINT (8.834359450000001 52.9834367)
...,...,...,...,...,...,...,...,...
12627617,12627617,100mN32134E42214,6098875,2,Paare ohne Kind(er),18 - 29,Weiblich,POINT (8.549510265037483 52.022315405216844)
12627618,12627618,100mN32134E42214,6098875,2,Paare ohne Kind(er),30 - 49,Männlich,POINT (8.549510265037483 52.022315405216844)
12627619,12627619,100mN32134E42214,6098876,1,Einpersonenhaushalte (Singlehaushalte),30 - 49,Weiblich,POINT (8.549510265037483 52.022315405216844)
12627620,12627620,100mN32134E42214,6098877,1,Einpersonenhaushalte (Singlehaushalte),30 - 49,Weiblich,POINT (8.549510265037483 52.022315405216844)


In [15]:
# get bounds from osm data or set manually
# extract buildings from osm data using extract_buildings_from_pbf_to_osmxml
buildings_path = "~/GeoData/OSM/Buildings/buildings_weser_bergland.osm"

In [12]:
result = extract_buildings_from_pbf_to_osmxml(osm_data_path, output_file_path=buildings_path)
# load buildings using geopandas

In [13]:
result

CompletedProcess(args='osmium tags-filter /Users/kaub/GeoData/OSM/weser_bergland.osm_osmium.pbf w/building= -o ~/GeoData/OSM/Buildings/buildings_weser_bergland.osm --overwrite', returncode=0, stdout=b'', stderr=b'')

In [37]:
abs_building_path = os.path.expanduser(buildings_path)
abs_building_path

'/Users/kaub/GeoData/OSM/Buildings/buildings_weser_bergland.osm'

In [42]:
# get current working directory
cwd = Path.cwd()
cwd

PosixPath('/Users/kaub/PycharmProjects/css-geodata-service/css_geodata_service/affected_persons')

In [2]:
# Path to the osmconf.ini file
osm_conf_path = "osmconf.ini"  # Replace with the actual path to osmconf.ini

# Set the OSM_CONFIG_FILE environment variable
os.environ["OSM_CONFIG_FILE"] = osm_conf_path

In [8]:
fiona.supported_drivers

{'DXF': 'rw',
 'CSV': 'raw',
 'OpenFileGDB': 'raw',
 'ESRIJSON': 'r',
 'ESRI Shapefile': 'raw',
 'FlatGeobuf': 'raw',
 'GeoJSON': 'raw',
 'GeoJSONSeq': 'raw',
 'GPKG': 'raw',
 'GML': 'rw',
 'OGR_GMT': 'rw',
 'GPX': 'rw',
 'MapInfo File': 'raw',
 'DGN': 'raw',
 'S57': 'r',
 'SQLite': 'raw',
 'TopoJSON': 'r'}

In [9]:
fiona.drvsupport.supported_drivers["OSM"] = "r"  # enable KML support which is disabled by default
fiona.drvsupport.supported_drivers["osm"] = "r"  # enable KML support which is disabled

In [10]:
fiona.supported_drivers

{'DXF': 'rw',
 'CSV': 'raw',
 'OpenFileGDB': 'raw',
 'ESRIJSON': 'r',
 'ESRI Shapefile': 'raw',
 'FlatGeobuf': 'raw',
 'GeoJSON': 'raw',
 'GeoJSONSeq': 'raw',
 'GPKG': 'raw',
 'GML': 'rw',
 'OGR_GMT': 'rw',
 'GPX': 'rw',
 'MapInfo File': 'raw',
 'DGN': 'raw',
 'S57': 'r',
 'SQLite': 'raw',
 'TopoJSON': 'r',
 'OSM': 'r',
 'osm': 'r'}

In [13]:
test = gpd.read_file("~/Downloads/map.osm", driver="OSM")

In [14]:
test.head()

,osm_id,osm_version,osm_timestamp,osm_uid,osm_user,osm_changeset,all_tags,geometry
0,10602431,14,2016-11-12 13:17:55+00:00,2739929,Erdrandbewohner,43581124,NaN,POINT (6.64423 49.75435)
1,10602432,5,2012-08-09 15:45:29+00:00,114405,Pajopath,12669974,NaN,POINT (6.64581 49.75383)
2,10602433,12,2011-08-19 21:38:19+00:00,114405,Pajopath,9069665,NaN,POINT (6.64665 49.75439)
3,10602434,8,2016-06-13 09:48:41+00:00,3052,Ole,39988489,NaN,POINT (6.64759 49.75541)
4,10602435,12,2018-01-27 18:07:49+00:00,2739929,Erdrandbewohner,55810814,NaN,POINT (6.64852 49.75634)


In [15]:
# todo apparently this does not work - ignore for now and load preexisting buildings
all_buildings = gpd.read_file("~/GeoData/OSM/Buildings/all_buildings_weser_bergland.osm")

ERROR:fiona._env:Too many features have accumulated in lines layer. Use the OGR_INTERLEAVED_READING=YES configuration option, or the INTERLEAVED_READING=YES open option, or the GDALDataset::GetNextFeature() / GDALDatasetGetNextFeature() API.


In [ ]:
len(all_buildings)

In [18]:
buildings = gpd.read_file("~/GeoData/OSM/residential_weser_bergland.gpkg")

In [16]:
len(buildings)

NameError: name 'buildings' is not defined

In [19]:
buildings.head()

,addr:city,addr:country,addr:housenumber,addr:housename,addr:postcode,addr:place,addr:street,email,name,opening_hours,...,source,start_date,wikipedia,id,timestamp,version,tags,osm_type,changeset,geometry
0,Herdecke,DE,9,NaN,58313,NaN,Harkortstraße,NaN,NaN,NaN,...,NaN,NaN,NaN,21602313,0,-1,NaN,way,NaN,"POLYGON ((7.42731 51.40072, 7.42715 51.40078, ..."
1,Herdecke,DE,3,NaN,58313,NaN,Rilkestraße,NaN,NaN,NaN,...,NaN,NaN,NaN,21602560,0,-1,NaN,way,NaN,"POLYGON ((7.42802 51.40119, 7.42808 51.40133, ..."
2,Herdecke,DE,15,NaN,58313,NaN,Rilkestraße,NaN,NaN,NaN,...,NaN,NaN,NaN,21602812,0,-1,NaN,way,NaN,"POLYGON ((7.42668 51.40140, 7.42675 51.40152, ..."
3,Herdecke,DE,16,NaN,58313,NaN,Harkortstraße,NaN,NaN,NaN,...,NaN,NaN,NaN,21602892,0,-1,NaN,way,NaN,"POLYGON ((7.42705 51.40108, 7.42712 51.40118, ..."
4,Herdecke,DE,18,NaN,58313,NaN,Harkortstraße,NaN,NaN,NaN,...,NaN,NaN,NaN,21602965,0,-1,NaN,way,NaN,"POLYGON ((7.42688 51.40125, 7.42682 51.40116, ..."


In [49]:
# reproject affected area to buildings crs
# affected_area = affected_area.to_crs(buildings.crs)

In [ ]:
population_geopandas = population_geopandas.to_crs(affected_area.crs)

In [50]:
affected_buildings = gpd.sjoin(buildings, affected_area, how="inner", op="intersects")

/Users/kaub/PycharmProjects/css-geodata-service/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3448: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


In [52]:
affected_buildings.head()

,addr:city,addr:country,addr:housenumber,addr:housename,addr:postcode,addr:place,addr:street,email,name,opening_hours,...,determinationMethod,endLifeSpanVersion,localId,namespace,versionId,qualitativeLikelihood,probabilityOfOccurrence,returnPeriod,assessmentMethod,magnitudeOrIntensity
92,Lügde,DE,2,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
93,Lügde,DE,4,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
94,Lügde,DE,8,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
95,Lügde,DE,8 a,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
96,Lügde,DE,10,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219809,Meinhard,DE,15,NaN,37276,NaN,Eschweger Straße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DEHE_3995_H-1,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
219810,Meinhard,DE,10,NaN,37276,NaN,Schwebdaer Straße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DEHE_3995_H-1,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
220540,Meinhard,DE,2a,NaN,37276,NaN,Eschweger Straße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DEHE_3998_H-1,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
222140,Eschwege,DE,2,NaN,37269,NaN,An der Berka,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DEHE_4002_H-1,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN


,addr:city,addr:country,addr:housenumber,addr:housename,addr:postcode,addr:place,addr:street,email,name,opening_hours,...,determinationMethod,endLifeSpanVersion,localId,namespace,versionId,qualitativeLikelihood,probabilityOfOccurrence,returnPeriod,assessmentMethod,magnitudeOrIntensity
92,Lügde,DE,2,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
93,Lügde,DE,4,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
94,Lügde,DE,8,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
95,Lügde,DE,8 a,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN
96,Lügde,DE,10,NaN,32676,NaN,Höxterstraße,NaN,NaN,NaN,...,indirectDetermination,NaN,nzHA_DENW_4_240_H-4,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.05,20,NaN,NaN


In [53]:
len(affected_buildings)

1380

In [66]:
# ignore buildings just look at households and persons
affected_population = gpd.sjoin(population_geopandas, affected_area, how="inner", op="intersects")

/Users/kaub/PycharmProjects/css-geodata-service/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3448: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):
/var/folders/ft/kkbmdx9s4fb0x94nztz0spp00000gn/T/ipykernel_47640/864890825.py:2: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  affected_population = gpd.sjoin(population_geopandas, affected_area, how="inner", op="intersects")


In [67]:
affected_population.head()

,person_id,cell_id,household_id,household_size,household_type,age,sex,geometry,index_right,gml_id,...,determinationMethod,endLifeSpanVersion,localId,namespace,versionId,qualitativeLikelihood,probabilityOfOccurrence,returnPeriod,assessmentMethod,magnitudeOrIntensity
10196,10196,100mN31058E42361,4949,3,Paare mit Kind(ern),18 - 29,Weiblich,POINT (8.79043 51.05779),2669,nzHA_DEHE_2194_H-1,...,indirectDetermination,NaN,nzHA_DEHE_2194_H-1,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.1,10,NaN,NaN
10197,10197,100mN31058E42361,4949,3,Paare mit Kind(ern),18 - 29,Männlich,POINT (8.79043 51.05779),2669,nzHA_DEHE_2194_H-1,...,indirectDetermination,NaN,nzHA_DEHE_2194_H-1,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.1,10,NaN,NaN
10198,10198,100mN31058E42361,4949,3,Paare mit Kind(ern),Unter 18,Männlich,POINT (8.79043 51.05779),2669,nzHA_DEHE_2194_H-1,...,indirectDetermination,NaN,nzHA_DEHE_2194_H-1,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.1,10,NaN,NaN
10199,10199,100mN31058E42361,4950,4,Paare mit Kind(ern),18 - 29,Weiblich,POINT (8.79043 51.05779),2669,nzHA_DEHE_2194_H-1,...,indirectDetermination,NaN,nzHA_DEHE_2194_H-1,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.1,10,NaN,NaN
10200,10200,100mN31058E42361,4950,4,Paare mit Kind(ern),30 - 49,Männlich,POINT (8.79043 51.05779),2669,nzHA_DEHE_2194_H-1,...,indirectDetermination,NaN,nzHA_DEHE_2194_H-1,https://registry.gdi-de.org/id/de.bund.bfg.ins...,NaN,high,0.1,10,NaN,NaN


In [68]:
len(affected_population)

47359

In [77]:
population_geopandas.head()

,person_id,cell_id,household_id,household_size,household_type,age,sex,geometry
0,0,100mN33211E42447,0,3,Paare mit Kind(ern),50 - 64,Weiblich,POINT (8.86498 52.99373)
1,1,100mN33211E42447,0,3,Paare mit Kind(ern),18 - 29,Männlich,POINT (8.86498 52.99373)
2,2,100mN33211E42447,0,3,Paare mit Kind(ern),Unter 18,Weiblich,POINT (8.86498 52.99373)
3,3,100mN33200E42427,1,3,Paare mit Kind(ern),18 - 29,Weiblich,POINT (8.83436 52.98344)
4,4,100mN33200E42427,1,3,Paare mit Kind(ern),30 - 49,Männlich,POINT (8.83436 52.98344)


In [79]:
population_geopandas_relevant_data = population_geopandas[["cell_id", "geometry"]]
population_geopandas_relevant_data.head()

,cell_id,geometry
0,100mN33211E42447,POINT (8.86498 52.99373)
1,100mN33211E42447,POINT (8.86498 52.99373)
2,100mN33211E42447,POINT (8.86498 52.99373)
3,100mN33200E42427,POINT (8.83436 52.98344)
4,100mN33200E42427,POINT (8.83436 52.98344)


In [80]:
# to increase speed first use only the unique locations / buildings and then filter the population
unique_locations_in_population = population_geopandas_relevant_data.drop_duplicates(subset=["geometry"])

In [81]:
unique_locations_in_population.head()

,cell_id,geometry
0,100mN33211E42447,POINT (8.86498 52.99373)
3,100mN33200E42427,POINT (8.83436 52.98344)
6,100mN33200E42427,POINT (8.83459 52.98368)
8,100mN33200E42427,POINT (8.83419 52.98364)
11,100mN33200E42427,POINT (8.83564 52.98344)


In [82]:
len(population_geopandas)

12627622

In [83]:
len(unique_locations_in_population)

2641628

In [86]:
logger = logging.getLogger(__name__)

In [ ]:
# try this using utils
filtered_population = filter_gdf_from_from_gdf_filter_own(unique_locations_in_population, affected_area)

INFO:css_geodata_service.extract.python_tools:Starting to filter data from polygons
INFO:css_geodata_service.extract.python_tools:Processing polygons 0.00% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 0.98% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 1.97% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 2.95% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 3.94% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 4.92% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 5.90% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 6.89% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 7.87% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 8.86% complete
INFO:css_geodata_service.extract.python_tools:Processing polygons 9.84% complete
INFO:css_geodata_service.

In [17]:
filtered_population.head()

NameError: name 'filtered_population' is not defined

In [18]:
len(filtered_population)

NameError: name 'filtered_population' is not defined